# VAW Phase 1 — Colab run: M9 behavior states, calibration, buildup videos and category report
Run notebooks 01 (M1/M2), 02 (M8a tracks) first. **No GPU needed here**: M9 reads the cached tracks in Drive.
What each step does and how to check it: `docs/VERIFY_M9.md`. Design and file formats: `docs/ARCHITECTURE.md`.

In [ ]:
from google.colab import drive, userdata
import os
drive.mount('/content/drive')
url = f"https://{userdata.get('GITHUB_TOKEN')}@github.com/KRITHIKSAI16/Violence-Against-Women.git"
if not os.path.isdir('/content/VAW'):
    !git clone -q {url} /content/VAW
else:
    !git -C /content/VAW pull -q {url} main
%cd /content/VAW
!pip install -q -r requirements.txt
!python -m pytest -q
!git log --oneline | head -3

## 1. M9 gate: behavior states for every clip
For every pair of tracked people: APPROACH, FOLLOW, HOVER, CORNER, ESCALATION, and the clips these flag as "needs deeper analysis".
Writes `VAW_results/gate/<Category>/<clip>_gate.json` + `_states.csv`, and `gate/proposals.json` (the hand-off to Phase II).
The table shows the share of clips per category with each behavior. **Normal is the false-alarm baseline.**

In [ ]:
!python -m src.assm.gate --config configs/colab.yaml

## 2. Calibration: do the thresholds make sense on the full corpus?
Prints (1) rates per category, (2) how long each behavior lasts in Normal vs other clips, (3) a sweep of each threshold showing the
flag rate on Normal (false alarms) vs the other categories, (4) suggested values. It changes nothing by itself.
This is in-sample tuning on the Normal label: treat it as a guide and confirm with the human review sheet (step 5).

In [ ]:
!python -m src.assm.calibrate --config configs/colab.yaml --workers 2

## 3. (Optional) Try different thresholds
Put values from the calibration table here, e.g. `SET = "follow_min_s=3 hover_min_s=2 hover_max_d=1.5"`, and run the cell.
They apply to this run only and are recorded inside every gate file. **Leave `SET` empty to restore the config defaults.**
If you settle on values, copy them into `configs/extran.yaml` and `configs/colab.yaml` (`gate:` block) in the repo and push.

In [ ]:
SET = ""   # e.g. "follow_min_s=3 hover_min_s=2"
!python -m src.assm.gate --config configs/colab.yaml --set {SET}

## 4. The deliverable: buildup videos, storyboards, category report
Renders a showcase (top-evidence + random clips per category, plus the review-sheet clips) and writes `VAW_results/report/index.html`,
`categories.csv`, `clips.csv`, `review_sheet.csv`. Videos stop BEFORE the detected escalation (see cut rules in `docs/VERIFY_M9.md`).
Add `--all` to render every clip (slow). Add `--set showcase_top=10 showcase_random=10` for a bigger showcase.

In [ ]:
!python -m src.report.buildup_report --config configs/colab.yaml --workers 2

## 5. Look at the results here

In [ ]:
import csv
from IPython.display import Image, Video, display
R = '/content/drive/MyDrive/VAW_results/report'
print(open(f'{R}/categories.csv').read())

In [ ]:
# Pick a clip from the showcase: change CLIP to any clip that has a video in clips.csv
rows = list(csv.DictReader(open(f'{R}/clips.csv')))
shown = [r for r in rows if r['video']]
print(len(shown), 'clips rendered. Flagged ones first:')
for r in sorted(shown, key=lambda r: -int(r['flag']))[:15]:
    print(f"  {r['clip_id']:<22}{r['category']:<16}{'FLAGGED' if r['flag']=='1' else '-':<9}{r['claim']}")

In [ ]:
CLIP = shown[0]['clip_id'] if shown else None
r = next(x for x in shown if x['clip_id'] == CLIP)
print(r['claim'], '| shows', r['cut_s'], 's:', r['cut_reason'])
display(Image(f"{R}/{r['storyboard']}"))
Video(f"{R}/{r['video']}", embed=True, width=520)

## 6. Human review sheet (the independent check)
Open `VAW_results/report/review_sheet.csv` (Drive -> right click -> Open with Google Sheets). For each clip play its video and fill **verdict**:
`correct` (the described behavior really happens), `partly`, `wrong` (it does not happen), or for unflagged clips `missed` (a buildup is visible that the system did not flag).
Count them: correct / flagged = precision of the gate by eye. Report this number with the table, not the in-sample rates.

To share the report: in Drive download the folder `VAW_results/report` (right click -> Download), unzip, open `index.html` in a browser.
Videos are linked relatively, so keep the folder structure.

## 7. (Optional) Render specific clips
Use this to produce the exact clips you will present. For a clip where the automatic cut is wrong, add a line `clip_id,cut_s` to
`configs/clip_overrides.csv` in the repo (manual cut time in seconds), push, `git pull` (re-run the first cell) and render again.

In [ ]:
!python -m src.report.buildup_video Stalking_v3 Kidnapping_v28 --config configs/colab.yaml